<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->


# Cosmos3 Action Policy with TensorRT-LLM

Build the trained DROID concatenated camera view from checked-in clips, then jointly predict a
16-step, 10D action chunk and a 17-frame rollout with `Cosmos3-Nano-Policy-DROID`.


## Start the TensorRT-LLM server

Follow the shared [TensorRT-LLM setup](../../README.md#tensorrt-llm-generator), then run this
from the TensorRT-LLM checkout. This notebook is a client; keep the server in a separate terminal.

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD}"
trtllm-serve nvidia/Cosmos3-Nano-Policy-DROID \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --port 8000
```

Install client-side decoding dependencies in this notebook kernel if needed:

```bash
pip install requests safetensors torch imageio imageio-ffmpeg pillow
```


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
ACTION_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "action"
OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_TRTLLM_ACTION_OUTPUT_ROOT", ACTION_ROOT / "outputs" / "notebooks" / "trt_llm")
).resolve()
TRTLLM_BASE_URL = os.environ.get("COSMOS3_TRTLLM_BASE_URL", "http://localhost:8000").rstrip("/")
TRTLLM_API_KEY = os.environ.get("COSMOS3_TRTLLM_API_KEY", "tensorrt_llm")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("COSMOS_ROOT:", COSMOS_ROOT)
print("ACTION_ROOT:", ACTION_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("TRTLLM_BASE_URL:", TRTLLM_BASE_URL)


## TensorRT-LLM response contract

The notebooks use the blocking `POST /v1/videos/generations` route. Every Action request sets
`format=safetensors`; the response is `application/octet-stream` with named `video`, `action`,
and `frame_rate` tensors. This differs from vLLM-Omni's top-level action metadata.

TensorRT-LLM also implements the asynchronous `POST /v1/videos` route. It returns HTTP 202,
can be polled through `GET /v1/videos/{id}`, and serves the same tensor payload from
`GET /v1/videos/{id}/content`.


In [ ]:
import json
import mimetypes
import time

import imageio.v3 as iio
import requests
from IPython.display import Video, display
from safetensors.torch import load as load_safetensors


def server_root_url() -> str:
    return TRTLLM_BASE_URL[:-3] if TRTLLM_BASE_URL.endswith("/v1") else TRTLLM_BASE_URL


def video_api_url() -> str:
    root = TRTLLM_BASE_URL if TRTLLM_BASE_URL.endswith("/v1") else f"{TRTLLM_BASE_URL}/v1"
    return f"{root}/videos/generations"


def wait_for_server(timeout_s: int = 1800, interval_s: int = 10) -> None:
    deadline = time.time() + timeout_s
    url = f"{server_root_url()}/health"
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=10)
            if response.ok:
                print("TensorRT-LLM server is ready:", url)
                return
        except requests.RequestException as exc:
            print("waiting for TensorRT-LLM:", exc)
        time.sleep(interval_s)
    raise TimeoutError(f"TensorRT-LLM did not become ready at {url}")


def submit_action(prompt: str, reference_path: Path, extra_params: dict, output_path: Path) -> dict:
    """Call the synchronous API and decode its action/video tensor payload."""
    reference_path = reference_path.resolve()
    if not reference_path.exists():
        raise FileNotFoundError(reference_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    media_type = mimetypes.guess_type(reference_path.name)[0] or "application/octet-stream"
    form = {
        "prompt": prompt,
        "format": "safetensors",
        "response_format": "url",
        "seed": "0",
        "extra_params": json.dumps(extra_params, separators=(",", ":")),
    }
    headers = {"Accept": "application/octet-stream"}
    if TRTLLM_API_KEY:
        headers["Authorization"] = f"Bearer {TRTLLM_API_KEY}"
    with reference_path.open("rb") as reference_file:
        response = requests.post(
            video_api_url(),
            data=form,
            files={"input_reference": (reference_path.name, reference_file, media_type)},
            headers=headers,
            timeout=3600,
        )
    if not response.ok:
        raise RuntimeError(f"TensorRT-LLM request failed ({response.status_code}): {response.text}")
    content_type = response.headers.get("content-type", "")
    if "application/octet-stream" not in content_type:
        raise RuntimeError(f"Expected a tensor payload, got {content_type!r}")
    output_path.write_bytes(response.content)
    payload = load_safetensors(response.content)
    missing = {"video", "action"} - payload.keys()
    if missing:
        raise RuntimeError(f"TensorRT-LLM action payload is missing {sorted(missing)}; got {sorted(payload)}")
    print("saved tensor payload:", output_path)
    print("video:", tuple(payload["video"].shape), payload["video"].dtype)
    print("action:", tuple(payload["action"].shape), payload["action"].dtype)
    return payload


def save_and_view_video(payload: dict, output_path: Path) -> Path:
    frames = payload["video"].detach().cpu().numpy()
    if frames.ndim == 5 and frames.shape[0] == 1:
        frames = frames[0]
    if frames.ndim != 4 or frames.shape[-1] != 3:
        raise ValueError(f"Expected video [T,H,W,3], got {frames.shape}")
    fps_tensor = payload.get("frame_rate")
    fps = float(fps_tensor.item()) if fps_tensor is not None else 24.0
    iio.imwrite(output_path, frames, fps=fps)
    print("saved rollout:", output_path, "fps:", fps)
    display(Video(str(output_path), embed=True))
    return output_path


## Build the DROID multiview first frame


In [ ]:
import subprocess

import imageio_ffmpeg
from IPython.display import Image as NotebookImage
from PIL import Image, ImageOps

DROID_ROOT = ACTION_ROOT / "assets" / "droid_lerobot_example"
camera_paths = {
    "wrist": DROID_ROOT / "videos/observation.image.wrist_image_left/chunk-000/file-000.mp4",
    "left": DROID_ROOT / "videos/observation.image.exterior_image_1_left/chunk-000/file-000.mp4",
    "right": DROID_ROOT / "videos/observation.image.exterior_image_2_left/chunk-000/file-000.mp4",
}
for path in camera_paths.values():
    if not path.exists():
        raise FileNotFoundError(path)

frame_dir = OUTPUT_ROOT / "policy_inputs"
frame_dir.mkdir(parents=True, exist_ok=True)
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
frames = {}
for name, video_path in camera_paths.items():
    frame_path = frame_dir / f"{name}.png"
    subprocess.run(
        [ffmpeg, "-y", "-loglevel", "error", "-i", str(video_path), "-frames:v", "1", str(frame_path)],
        check=True,
    )
    frames[name] = Image.open(frame_path).convert("RGB")

target_w, target_h = 640, 540
top_h = target_h // 2
bottom_h = target_h - top_h
half_w = target_w // 2
wrist = ImageOps.fit(frames["wrist"], (target_w, top_h), method=Image.Resampling.BICUBIC)
left = ImageOps.fit(frames["left"], (half_w, bottom_h), method=Image.Resampling.BICUBIC)
right = ImageOps.fit(frames["right"], (half_w, bottom_h), method=Image.Resampling.BICUBIC)
conditioning = Image.new("RGB", (target_w, target_h))
conditioning.paste(wrist, (0, 0))
conditioning.paste(left, (0, top_h))
conditioning.paste(right, (half_w, top_h))
policy_image_path = frame_dir / "droid_policy_first_frame.png"
conditioning.save(policy_image_path)
display(NotebookImage(filename=str(policy_image_path), width=640))

prompt = os.environ.get(
    "COSMOS3_POLICY_PROMPT",
    "Pick up the object and place it in the target container.",
)
extra_params = {
    "action_mode": "policy",
    "domain_name": "droid_lerobot",
    "view_point": "concat_view",
    "use_guardrails": True,
}
print("prompt:", prompt)
print(json.dumps(extra_params, indent=2))


## Run policy inference


In [ ]:
wait_for_server()
policy_payload = submit_action(
    prompt,
    policy_image_path,
    extra_params,
    OUTPUT_ROOT / "policy_droid.safetensors",
)


## Inspect the rollout and predicted action chunk


In [ ]:
save_and_view_video(policy_payload, OUTPUT_ROOT / "policy_droid.mp4")
policy_action = policy_payload["action"].detach().cpu().numpy()
print("predicted action shape:", policy_action.shape)
print("first five predicted action rows:")
print(policy_action[:5])
